# 02 · Finite CSPs, quantified strategies, and bounded Go

## Purpose and scope

This notebook provides exact, deliberately small reference tasks for native AHR.
It does not train a model or solve unrestricted Go. Ordinary CSP finds one
satisfying assignment. Quantified CSP checks a contingent strategy against every
universally quantified reply. The Go evaluator includes **every legal opponent
move** and records the precise horizon and objective of each result.

**Inputs:** Python 3.10+, `nbformat`, and notebook 01 in this directory.
Notebook 00 checks the shared environment; notebook 01 defines Go rules.
**Outputs:** reusable tagged definitions and `reports/02_encoding_validation.json`.
**Expected runtime:** under two minutes on CPU for these small validation cases;
no GPU is needed. Default search limits are node budgets, never silent labels.
**Resume:** this is a short, deterministic validation job, with no training
checkpoint. Restart the kernel and Run All to reproduce it from the beginning.
The report is replaced atomically only after all checks pass. Future long-running
generation belongs in notebook 03 and must implement its own validated resume.

## Meaning of labels

- `SAT`: a witness/strategy establishes the explicitly stated finite objective.
- `UNSAT`: the finite search disproved that objective; for certificate checking,
  it means the supplied certificate is invalid.
- `UNKNOWN`: a budget ran out, or terminal Go victory was unresolved at the
  horizon. It is never converted into a negative training example.

`area_at_horizon` is a deterministic bounded score proxy. It is not a claim of
winning the eventual game. `terminal_win` accepts only actual terminal victory;
nonterminal horizon leaves remain unknown. Score uses notebook 01's specified
area rules and komi. The exact full state, including superko history, is retained.

In [ ]:
from pathlib import Path
import json
import os
import nbformat

PROJECT_ROOT = Path(os.environ.get("AHR_GO_PROJECT", str(Path.cwd()))).expanduser().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif (PROJECT_ROOT / "remote_notebooks/ahr_go/notebooks").is_dir():
    PROJECT_ROOT = PROJECT_ROOT / "remote_notebooks/ahr_go"
NOTEBOOK_DIR = PROJECT_ROOT / "notebooks"
REPORT_DIR = PROJECT_ROOT / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
rules_path = NOTEBOOK_DIR / "01_go_rules_and_independent_verifier.ipynb"
if not rules_path.exists():
    raise FileNotFoundError("Required rules notebook is missing: " + str(rules_path))
rules_notebook = nbformat.read(rules_path, as_version=4)
loaded_rules_cells = 0
for cell in rules_notebook.cells:
    if cell.cell_type == "code" and "ahr-export" in cell.metadata.get("tags", []):
        exec(compile(cell.source, str(rules_path), "exec"), globals())
        loaded_rules_cells += 1
if not loaded_rules_cells:
    raise RuntimeError("Notebook 01 has no ahr-export definitions")
print(f"Loaded {loaded_rules_cells} definition cells from {rules_path.name}.")

## 1. A small, serializable constraint language

Each variable has a finite, nonempty integer domain. A table constraint explicitly
lists its allowed tuples; conjunction combines constraints. Boolean SAT clauses,
graph coloring, and small arithmetic relations can all use this format. An empty
allowed table means false. A zero-variable constraint with `[[]]` means true.
The implementation rejects malformed domains, scopes, prefixes and row values.

For QCSP, a prefix contains each variable exactly once, for example
`(('exists','x'), ('forall','y'), ('exists','z'))`. Existential choices may depend
on earlier replies, never on later variables. Only definition cells carry the
`ahr-export` tag, so downstream imports cannot trigger workflows or tests.

In [ ]:
from dataclasses import dataclass
from itertools import product
from math import isfinite

SAT, UNSAT, UNKNOWN = "SAT", "UNSAT", "UNKNOWN"


@dataclass(frozen=True)
class TableConstraint:
    scope: tuple[str, ...]
    allowed: tuple[tuple[int, ...], ...]


@dataclass(frozen=True)
class CSP:
    domains: dict[str, tuple[int, ...]]
    constraints: tuple[TableConstraint, ...]
    prefix: tuple[tuple[str, str], ...] = ()


@dataclass
class SolveResult:
    status: str
    witness: dict | None = None
    strategy: dict | None = None
    nodes: int = 0
    scope: str = ""
    reason: str = ""


class _NodeBudget:
    def __init__(self, maximum):
        if type(maximum) is not int or maximum < 0:
            raise ValueError("Budget must be a nonnegative integer")
        self.maximum = maximum
        self.nodes = 0

    def tick(self):
        if self.nodes >= self.maximum:
            return False
        self.nodes += 1
        return True


def validate_csp(problem, quantified=False):
    if not isinstance(problem, CSP):
        raise TypeError("Expected CSP")
    for name, domain in problem.domains.items():
        if not isinstance(name, str) or not name:
            raise ValueError("Variable names must be nonempty strings")
        if not domain or any(type(value) is not int for value in domain):
            raise ValueError("Domains must be nonempty integer sequences")
        if len(set(domain)) != len(domain):
            raise ValueError("Domain values must be unique")
    for constraint in problem.constraints:
        if len(set(constraint.scope)) != len(constraint.scope):
            raise ValueError("A constraint scope cannot repeat a variable")
        if any(name not in problem.domains for name in constraint.scope):
            raise ValueError("Unknown variable in constraint")
        for row in constraint.allowed:
            if len(row) != len(constraint.scope):
                raise ValueError("Table row arity mismatch")
            if any(type(value) is not int or value not in problem.domains[name]
                   for name, value in zip(constraint.scope, row)):
                raise ValueError("Table value outside domain")
    if problem.prefix or quantified:
        names = []
        for entry in problem.prefix:
            if len(entry) != 2:
                raise ValueError("Prefix entries are (quantifier, variable)")
            quantifier, name = entry
            if quantifier not in ("exists", "forall"):
                raise ValueError("Quantifier must be exists or forall")
            names.append(name)
        if len(names) != len(set(names)) or set(names) != set(problem.domains):
            raise ValueError("Prefix must contain each variable exactly once")
    return problem


def csp_to_dict(problem):
    validate_csp(problem)
    return {
        "schema": "ahr_finite_table_csp_v1",
        "domains": {key: list(value) for key, value in problem.domains.items()},
        "constraints": [{"scope": list(c.scope),
                         "allowed": [list(row) for row in c.allowed]}
                        for c in problem.constraints],
        "prefix": [list(entry) for entry in problem.prefix],
    }


def csp_from_dict(data):
    if data.get("schema", "ahr_finite_table_csp_v1") != "ahr_finite_table_csp_v1":
        raise ValueError("Unsupported CSP schema")
    instance = CSP(
        {key: tuple(value) for key, value in data["domains"].items()},
        tuple(TableConstraint(tuple(c["scope"]), tuple(tuple(row) for row in c["allowed"]))
              for c in data["constraints"]),
        tuple(tuple(entry) for entry in data.get("prefix", ())),
    )
    return validate_csp(instance)


def verify_assignment(problem, assignment):
    validate_csp(problem)
    if not isinstance(assignment, dict) or set(assignment) != set(problem.domains):
        return False
    if any(type(assignment[name]) is not int or assignment[name] not in domain
           for name, domain in problem.domains.items()):
        return False
    return all(tuple(assignment[name] for name in constraint.scope) in constraint.allowed
               for constraint in problem.constraints)


def _partial_feasible(problem, assignment):
    return all(any(all(name not in assignment or assignment[name] == value
                       for name, value in zip(constraint.scope, row))
                   for row in constraint.allowed)
               for constraint in problem.constraints)

## 2. Existential search and a separate exhaustive reference

The search solver prunes partial assignments when a table has no compatible row.
The reference instead builds complete Cartesian-product assignments and evaluates
the full tables directly. Thus small-instance agreement checks different search
paths; SAT certificates are additionally checked without running the solver.

In [ ]:
def solve_csp(problem, max_nodes=100_000):
    validate_csp(problem)
    if any(q == "forall" for q, _ in problem.prefix):
        raise ValueError("Use solve_qcsp for a universal prefix")
    budget = _NodeBudget(max_nodes)
    order = sorted(problem.domains, key=lambda name: (len(problem.domains[name]), name))

    def visit(depth, assignment):
        if not budget.tick():
            return UNKNOWN, None
        if not _partial_feasible(problem, assignment):
            return UNSAT, None
        if depth == len(order):
            return SAT, dict(assignment)
        name = order[depth]
        unknown = False
        for value in problem.domains[name]:
            assignment[name] = value
            status, witness = visit(depth + 1, assignment)
            del assignment[name]
            if status == SAT:
                return SAT, witness
            unknown |= status == UNKNOWN
        return (UNKNOWN if unknown else UNSAT), None

    status, witness = visit(0, {})
    return SolveResult(status, witness=witness, nodes=budget.nodes, scope="ordinary_csp",
                       reason="node_budget" if status == UNKNOWN else "")


def reference_csp(problem, max_assignments=100_000):
    validate_csp(problem)
    if any(q == "forall" for q, _ in problem.prefix):
        raise ValueError("Use reference_qcsp for a universal prefix")
    budget = _NodeBudget(max_assignments)
    names = list(problem.domains)
    indices = {name: i for i, name in enumerate(names)}
    for values in product(*(problem.domains[name] for name in names)):
        if not budget.tick():
            return SolveResult(UNKNOWN, nodes=budget.nodes, scope="ordinary_csp_reference",
                               reason="assignment_budget")
        accepted = True
        for constraint in problem.constraints:
            row = tuple(values[indices[name]] for name in constraint.scope)
            if not any(row == allowed for allowed in constraint.allowed):
                accepted = False
                break
        if accepted:
            return SolveResult(SAT, witness=dict(zip(names, values)), nodes=budget.nodes,
                               scope="ordinary_csp_reference")
    return SolveResult(UNSAT, nodes=budget.nodes, scope="ordinary_csp_reference")

## 3. Alternating quantifiers and strategy certificates

An existential node stores one choice. A universal node must store one child for
**every** value in its domain, exactly once. The independent checker reconstructs
the prefix assignment and checks each leaf's constraints. It rejects a missing
reply, a choice of the wrong variable, or a witness that sees a future reply.
The exhaustive reference evaluates the complete truth table and reduces it from
the final quantifier backwards; it does not reuse the solver or its pruning.

In [ ]:
def solve_qcsp(problem, max_nodes=100_000):
    validate_csp(problem, quantified=True)
    budget = _NodeBudget(max_nodes)

    def visit(depth, assignment):
        if not budget.tick():
            return UNKNOWN, None
        if not _partial_feasible(problem, assignment):
            return UNSAT, None
        if depth == len(problem.prefix):
            return SAT, {"kind": "leaf"}
        quantifier, name = problem.prefix[depth]
        children, unknown = [], False
        for value in problem.domains[name]:
            assignment[name] = value
            status, child = visit(depth + 1, assignment)
            del assignment[name]
            if quantifier == "exists" and status == SAT:
                return SAT, {"kind": "exists", "var": name, "value": value, "child": child}
            if quantifier == "forall" and status == UNSAT:
                return UNSAT, None
            unknown |= status == UNKNOWN
            children.append({"value": value, "child": child})
        if unknown:
            return UNKNOWN, None
        if quantifier == "exists":
            return UNSAT, None
        return SAT, {"kind": "forall", "var": name, "children": children}

    status, strategy = visit(0, {})
    return SolveResult(status, strategy=strategy, nodes=budget.nodes, scope="quantified_csp",
                       reason="node_budget" if status == UNKNOWN else "")


def reference_qcsp(problem, max_assignments=100_000):
    validate_csp(problem, quantified=True)
    budget = _NodeBudget(max_assignments)
    names = [name for _, name in problem.prefix]
    index = {name: i for i, name in enumerate(names)}
    truth = {}
    for values in product(*(problem.domains[name] for name in names)):
        if not budget.tick():
            return SolveResult(UNKNOWN, nodes=budget.nodes, scope="quantified_csp_reference",
                               reason="assignment_budget")
        truth[values] = all(
            tuple(values[index[name]] for name in c.scope) in set(c.allowed)
            for c in problem.constraints
        )
    for depth in range(len(names) - 1, -1, -1):
        quantifier, name = problem.prefix[depth]
        reduced = {}
        for earlier in product(*(problem.domains[n] for n in names[:depth])):
            branches = [truth[earlier + (value,)] for value in problem.domains[name]]
            reduced[earlier] = any(branches) if quantifier == "exists" else all(branches)
        truth = reduced
    return SolveResult(SAT if truth[()] else UNSAT, nodes=budget.nodes,
                       scope="quantified_csp_reference")


def verify_qcsp_strategy(problem, strategy, max_nodes=100_000):
    validate_csp(problem, quantified=True)
    budget = _NodeBudget(max_nodes)

    def check(depth, assignment, node):
        if not budget.tick():
            return UNKNOWN
        if not isinstance(node, dict):
            return UNSAT
        if depth == len(problem.prefix):
            if node != {"kind": "leaf"}:
                return UNSAT
            for constraint in problem.constraints:
                values = tuple(assignment[name] for name in constraint.scope)
                if values not in constraint.allowed:
                    return UNSAT
            return SAT
        quantifier, name = problem.prefix[depth]
        if node.get("kind") != quantifier or node.get("var") != name:
            return UNSAT
        if quantifier == "exists":
            value = node.get("value")
            if type(value) is not int or value not in problem.domains[name]:
                return UNSAT
            return check(depth + 1, {**assignment, name: value}, node.get("child"))
        children = node.get("children")
        if not isinstance(children, list) or len(children) != len(problem.domains[name]):
            return UNSAT
        if any(not isinstance(child, dict) or type(child.get("value")) is not int
               for child in children):
            return UNSAT
        values = [child["value"] for child in children]
        if len(set(values)) != len(values) or set(values) != set(problem.domains[name]):
            return UNSAT
        statuses = [check(depth + 1, {**assignment, name: child["value"]}, child.get("child"))
                    for child in children]
        if UNSAT in statuses:
            return UNSAT
        return UNKNOWN if UNKNOWN in statuses else SAT

    status = check(0, {}, strategy)
    return SolveResult(status, nodes=budget.nodes, scope="qcsp_certificate_check",
                       reason="node_budget" if status == UNKNOWN else
                       ("invalid_certificate" if status == UNSAT else ""))

## 4. Go as a bounded adversarial constraint task

These search functions instantiate an AND/OR constraint tree directly from legal
Go transitions. They do not construct a giant static table of all possible full
games. At our turn, one legal move suffices; at the opponent's turn, every legal
reply must satisfy the objective. The complete state is passed through notebook
01, including the history needed for positional superko.

An existential certificate stores a chosen move; an opponent node stores all
legal replies. The checker enumerates the authoritative legal-action list again
and verifies each branch separately. The exhaustive reference explores every
branch even when the outcome is already determined. No heuristic value replaces
an unknown leaf. Early terminal states use exact area score.

In [ ]:
@dataclass(frozen=True)
class GoObjective:
    kind: str = "area_at_horizon"
    player: int = 1
    threshold: float = 0.0


def _validate_go_objective(horizon, objective):
    if type(horizon) is not int or horizon < 0:
        raise ValueError("Horizon must be a nonnegative integer")
    if objective.kind not in ("area_at_horizon", "terminal_win"):
        raise ValueError("Unknown Go objective")
    if type(objective.player) is not int or objective.player not in (1, -1):
        raise ValueError("Objective player must be BLACK=1 or WHITE=-1")
    if not isfinite(objective.threshold):
        raise ValueError("Threshold must be finite")


def _go_scope(horizon, objective):
    return (f"go:{objective.kind}:horizon={horizon}:player={objective.player}:"
            f"threshold={objective.threshold}")


def _go_leaf_status(state, depth, objective):
    if not state.terminal and depth > 0:
        return None
    if objective.kind == "terminal_win":
        if not state.terminal:
            return UNKNOWN
        return SAT if objective.player * area_score(state) > objective.threshold else UNSAT
    return SAT if objective.player * area_score(state) >= objective.threshold else UNSAT


def solve_go_bounded(state, horizon, objective, max_nodes=100_000):
    _validate_go_objective(horizon, objective)
    budget = _NodeBudget(max_nodes)

    def visit(position, depth):
        if not budget.tick():
            return UNKNOWN, None
        leaf = _go_leaf_status(position, depth, objective)
        if leaf is not None:
            return leaf, {"kind": "leaf"} if leaf == SAT else None
        own_turn = position.to_play == objective.player
        children, unknown = [], False
        for action in legal_actions(position):
            status, child = visit(play(position, action), depth - 1)
            if own_turn and status == SAT:
                return SAT, {"kind": "move", "action": action, "child": child}
            if not own_turn and status == UNSAT:
                return UNSAT, None
            unknown |= status == UNKNOWN
            children.append({"action": action, "child": child})
        if unknown:
            return UNKNOWN, None
        if own_turn:
            return UNSAT, None
        return SAT, {"kind": "replies", "children": children}

    status, strategy = visit(state, horizon)
    return SolveResult(status, strategy=strategy, nodes=budget.nodes,
                       scope=_go_scope(horizon, objective),
                       reason="budget_or_nonterminal_horizon" if status == UNKNOWN else "")


def reference_go_bounded(state, horizon, objective, max_nodes=100_000):
    _validate_go_objective(horizon, objective)
    budget = _NodeBudget(max_nodes)

    def evaluate(position, remaining):
        if not budget.tick():
            return UNKNOWN
        if position.terminal:
            signed_score = area_score(position) * objective.player
            succeeds = (signed_score > objective.threshold if objective.kind == "terminal_win"
                        else signed_score >= objective.threshold)
            return SAT if succeeds else UNSAT
        if remaining == 0:
            if objective.kind == "terminal_win":
                return UNKNOWN
            return SAT if area_score(position) * objective.player >= objective.threshold else UNSAT
        outcomes = [evaluate(play(position, action), remaining - 1)
                    for action in legal_actions(position)]
        if position.to_play == objective.player:
            if SAT in outcomes:
                return SAT
            return UNKNOWN if UNKNOWN in outcomes else UNSAT
        if UNSAT in outcomes:
            return UNSAT
        return UNKNOWN if UNKNOWN in outcomes else SAT

    status = evaluate(state, horizon)
    return SolveResult(status, nodes=budget.nodes, scope=_go_scope(horizon, objective),
                       reason="budget_or_nonterminal_horizon" if status == UNKNOWN else "")


def verify_go_strategy(state, horizon, objective, strategy, max_nodes=100_000):
    _validate_go_objective(horizon, objective)
    budget = _NodeBudget(max_nodes)

    def check(position, remaining, node):
        if not budget.tick():
            return UNKNOWN
        if not isinstance(node, dict):
            return UNSAT
        if position.terminal or remaining == 0:
            if node != {"kind": "leaf"}:
                return UNSAT
            if objective.kind == "terminal_win" and not position.terminal:
                return UNKNOWN
            signed_score = objective.player * area_score(position)
            accepted = (signed_score > objective.threshold if objective.kind == "terminal_win"
                        else signed_score >= objective.threshold)
            return SAT if accepted else UNSAT
        actions = list(legal_actions(position))
        if position.to_play == objective.player:
            action = node.get("action")
            if node.get("kind") != "move" or type(action) is not int or action not in actions:
                return UNSAT
            return check(play(position, action), remaining - 1, node.get("child"))
        children = node.get("children")
        if node.get("kind") != "replies" or not isinstance(children, list):
            return UNSAT
        if any(not isinstance(child, dict) or type(child.get("action")) is not int
               for child in children):
            return UNSAT
        proposed = [child["action"] for child in children]
        if len(proposed) != len(actions) or len(set(proposed)) != len(proposed):
            return UNSAT
        if set(proposed) != set(actions):
            return UNSAT
        statuses = [check(play(position, child["action"]), remaining - 1, child.get("child"))
                    for child in children]
        if UNSAT in statuses:
            return UNSAT
        return UNKNOWN if UNKNOWN in statuses else SAT

    status = check(state, horizon, strategy)
    return SolveResult(status, nodes=budget.nodes,
                       scope="certificate_check:" + _go_scope(horizon, objective),
                       reason="budget_or_nonterminal_horizon" if status == UNKNOWN else
                       ("invalid_certificate" if status == UNSAT else ""))

## 5. Validation: a winning line can fail against an opponent

For two Boolean variables, require `x == y`. There exists a satisfying assignment.
But `exists x, forall y: x == y` is false: the opponent chooses the other value.
Reversing the order to `forall y, exists x` is true and requires a response that
depends on the observed `y`. This tests both adversarial quantification and
contingent strategy structure.

In [ ]:
from copy import deepcopy
from datetime import datetime, timezone
import hashlib
import random
import time

validation_start = time.monotonic()
validation = {"schema": "ahr_encoding_validation_v1", "status": "running", "checks": {}}
equality = TableConstraint(("x", "y"), ((0, 0), (1, 1)))
ordinary = CSP({"x": (0, 1), "y": (0, 1)}, (equality,))
wrong_order = CSP(ordinary.domains, ordinary.constraints, (("exists", "x"), ("forall", "y")))
responsive = CSP(ordinary.domains, ordinary.constraints, (("forall", "y"), ("exists", "x")))
ordinary_result = solve_csp(ordinary)
wrong_result = solve_qcsp(wrong_order)
responsive_result = solve_qcsp(responsive)
assert ordinary_result.status == SAT and verify_assignment(ordinary, ordinary_result.witness)
assert wrong_result.status == UNSAT and reference_qcsp(wrong_order).status == UNSAT
assert responsive_result.status == SAT and reference_qcsp(responsive).status == SAT
assert verify_qcsp_strategy(responsive, responsive_result.strategy).status == SAT
missing_reply = deepcopy(responsive_result.strategy)
missing_reply["children"].pop()
assert verify_qcsp_strategy(responsive, missing_reply).status == UNSAT
wrong_choice = deepcopy(responsive_result.strategy)
wrong_choice["children"][0]["child"]["value"] = 1
assert verify_qcsp_strategy(responsive, wrong_choice).status == UNSAT
assert csp_to_dict(csp_from_dict(csp_to_dict(responsive))) == csp_to_dict(responsive)
validation["checks"]["quantifier_counterexample"] = {
    "ordinary_exists_exists": ordinary_result.status,
    "exists_forall": wrong_result.status,
    "forall_exists": responsive_result.status,
    "missing_reply_rejected": True,
    "wrong_response_rejected": True,
    "json_roundtrip": True,
}
print("Equality task: ordinary SAT; exists/forall UNSAT; forall/exists SAT.")
print("Independent strategy checker rejected omitted and incorrect opponent responses.")

## 6. Exhaustive Boolean matrices and finite-domain edge cases

Every two-variable Boolean relation (16 truth tables), all four quantifier
patterns, and both variable orders are checked against the exhaustive reference.
Small seeded multi-constraint problems provide additional coverage. This is a
software correctness test, not a held-out learning benchmark.

In [ ]:
ordinary_cases = 0
quantified_cases = 0
certificates_checked = 0
for mask in range(16):
    rows = tuple(row for index, row in enumerate(product((0, 1), repeat=2))
                 if mask & (1 << index))
    base = CSP({"x": (0, 1), "y": (0, 1)}, (TableConstraint(("x", "y"), rows),))
    found = solve_csp(base)
    exact = reference_csp(base)
    assert found.status == exact.status
    if found.status == SAT:
        assert verify_assignment(base, found.witness)
    ordinary_cases += 1
    for order in (("x", "y"), ("y", "x")):
        for quantifiers in product(("exists", "forall"), repeat=2):
            instance = CSP(base.domains, base.constraints, tuple(zip(quantifiers, order)))
            found = solve_qcsp(instance)
            exact = reference_qcsp(instance)
            assert found.status == exact.status, (mask, instance.prefix)
            if found.status == SAT:
                assert verify_qcsp_strategy(instance, found.strategy).status == SAT
                certificates_checked += 1
            quantified_cases += 1

rng = random.Random(20260923)
for trial in range(80):
    names = ("a", "b", "c", "d")
    domains = {name: (0, 1) for name in names}
    constraints = []
    for _ in range(4):
        scope = tuple(rng.sample(list(names), 3))
        allowed = tuple(row for row in product((0, 1), repeat=3) if rng.random() < 0.7)
        constraints.append(TableConstraint(scope, allowed))
    base = CSP(domains, tuple(constraints))
    found = solve_csp(base)
    assert found.status == reference_csp(base).status
    if found.status == SAT:
        assert verify_assignment(base, found.witness)
    ordinary_cases += 1
    prefix = tuple((rng.choice(("exists", "forall")), name) for name in names)
    instance = CSP(domains, tuple(constraints), prefix)
    found = solve_qcsp(instance)
    assert found.status == reference_qcsp(instance).status
    if found.status == SAT:
        assert verify_qcsp_strategy(instance, found.strategy).status == SAT
        certificates_checked += 1
    quantified_cases += 1

ternary = CSP({"u": (0, 1, 2), "v": (0, 1, 2)},
              (TableConstraint(("u", "v"), ((0, 1), (1, 2), (2, 0))),),
              (("forall", "u"), ("exists", "v")))
assert solve_qcsp(ternary).status == reference_qcsp(ternary).status == SAT
assert verify_qcsp_strategy(ternary, solve_qcsp(ternary).strategy).status == SAT
for empty, expected in ((CSP({}, ()), SAT), (CSP({}, (TableConstraint((), ()),)), UNSAT)):
    assert solve_csp(empty).status == reference_csp(empty).status == expected
    assert solve_qcsp(empty).status == reference_qcsp(empty).status == expected
for function, instance in ((solve_csp, ordinary), (solve_qcsp, responsive)):
    assert function(instance, max_nodes=0).status == UNKNOWN
    assert function(instance, max_nodes=1).status == UNKNOWN
    full_status = function(instance).status
    for budget in range(2, 12):
        limited = function(instance, max_nodes=budget)
        assert limited.status in (UNKNOWN, full_status)
assert reference_csp(ordinary, max_assignments=0).status == UNKNOWN
assert reference_qcsp(responsive, max_assignments=0).status == UNKNOWN
assert verify_qcsp_strategy(responsive, responsive_result.strategy, max_nodes=0).status == UNKNOWN
assert not verify_assignment(ordinary, {"x": 0})
assert not verify_assignment(ordinary, {"x": 0, "y": 0, "extra": 0})
validation["checks"]["csp_reference_comparison"] = {
    "ordinary_instances": ordinary_cases,
    "quantified_instances": quantified_cases,
    "quantified_sat_certificates": certificates_checked,
    "finite_ternary_domain": True,
    "zero_variable_truth_and_falsehood": True,
    "budget_unknown_preserved": True,
}
print(f"Matched independent references on {ordinary_cases} ordinary and {quantified_cases} quantified instances.")

## 7. Bounded Go cross-checks and complete opponent coverage

Enumerate every legal 2×2 history through two plies and compare bounded search
with exhaustive minimax for both player objectives. These deliberately tiny
boards exercise the actual transition rules while keeping complete enumeration
cheap. Tests also reject a certificate that omits passing, preserve full history,
and distinguish a deterministic horizon score from an unresolved terminal win.

In [ ]:
start = initial_state(size=2, komi=0.5)
frontier, states = [start], [start]
for _ in range(2):
    following = []
    for state in frontier:
        if not state.terminal:
            following.extend(play(state, action) for action in legal_actions(state))
    states.extend(following)
    frontier = following
go_cases = 0
go_certificates = 0
for state in states:
    for player in (BLACK, WHITE):
        objectives = [GoObjective("area_at_horizon", player, threshold)
                      for threshold in (-1.0, 1.0)]
        objectives.append(GoObjective("terminal_win", player, 0.0))
        for objective in objectives:
            found = solve_go_bounded(state, 2, objective)
            exact = reference_go_bounded(state, 2, objective)
            assert found.status == exact.status, (state, objective)
            if found.status == SAT:
                assert verify_go_strategy(state, 2, objective, found.strategy).status == SAT
                go_certificates += 1
            go_cases += 1

opponent_start = play(start, start.pass_action)
easy_objective = GoObjective("area_at_horizon", BLACK, -100.0)
all_replies = solve_go_bounded(opponent_start, 1, easy_objective)
assert all_replies.status == SAT and all_replies.strategy["kind"] == "replies"
assert {entry["action"] for entry in all_replies.strategy["children"]} == set(legal_actions(opponent_start))
missing_pass = deepcopy(all_replies.strategy)
missing_pass["children"] = [entry for entry in missing_pass["children"]
                            if entry["action"] != opponent_start.pass_action]
assert verify_go_strategy(opponent_start, 1, easy_objective, missing_pass).status == UNSAT
assert solve_go_bounded(start, 2, easy_objective, max_nodes=0).status == UNKNOWN
assert solve_go_bounded(start, 2, easy_objective, max_nodes=1).status == UNKNOWN
assert verify_go_strategy(opponent_start, 1, easy_objective,
                          all_replies.strategy, max_nodes=0).status == UNKNOWN
assert solve_go_bounded(start, 0, GoObjective("terminal_win", BLACK)).status == UNKNOWN
assert reference_go_bounded(start, 0, GoObjective("terminal_win", BLACK)).status == UNKNOWN
assert verify_go_strategy(start, 0, GoObjective("terminal_win", BLACK),
                          {"kind": "leaf"}).status == UNKNOWN
terminal = play(opponent_start, opponent_start.pass_action)
assert terminal.terminal
assert solve_go_bounded(terminal, 0, GoObjective("terminal_win", BLACK)).status == UNSAT
assert solve_go_bounded(terminal, 0, GoObjective("terminal_win", WHITE)).status == SAT
original_history = start.history
solve_go_bounded(start, 2, easy_objective)
assert start.history == original_history
validation["checks"]["go_reference_comparison"] = {
    "reachable_histories_through_two_plies": len(states),
    "bounded_instances": go_cases,
    "sat_certificates_checked": go_certificates,
    "every_legal_opponent_reply_included": True,
    "missing_pass_certificate_rejected": True,
    "nonterminal_cutoff_unknown": True,
    "node_budget_unknown": True,
    "terminal_score_both_colors": True,
    "input_history_unchanged": True,
}
print(f"Matched exhaustive Go evaluation on {go_cases} bounded tasks from {len(states)} histories.")
print("Omitted-pass proof rejected; terminal-cutoff and zero-budget cases remain UNKNOWN.")

## 8. Write the validation record

The record contains exact case counts, rules notebook provenance, and the
statement of what was checked. It makes no claim about AHR training, learned
solver quality, full-game proofs, or 9×9/19×19 playing strength.

In [ ]:
validation.update({
    "status": "passed",
    "completed_utc": datetime.now(timezone.utc).isoformat(),
    "runtime_seconds": round(time.monotonic() - validation_start, 4),
    "rules_notebook": rules_path.name,
    "rules_notebook_sha256": hashlib.sha256(rules_path.read_bytes()).hexdigest(),
    "scope": "Finite CSP/QCSP correctness and exhaustive tiny bounded Go checks; no model training",
    "go_objectives": {
        "area_at_horizon": "signed area >= threshold at terminal or horizon; bounded proxy only",
        "terminal_win": "terminal signed area > threshold; nonterminal horizon UNKNOWN",
    },
})
report_path = REPORT_DIR / "02_encoding_validation.json"
temporary = report_path.with_suffix(".json.tmp")
temporary.write_text(json.dumps(validation, indent=2) + "\n", encoding="utf-8")
os.replace(temporary, report_path)
print(f"PASS — wrote {report_path.relative_to(PROJECT_ROOT)}")
print(f"Validation took {validation['runtime_seconds']:.2f} seconds on CPU.")

## Next notebook

Notebook 03 can import only `ahr-export` cells from 01 and 02 using `nbformat`
and `exec` in dependency order. It can generate serialized table constraints,
solve small examples, check SAT witnesses/strategies independently, and retain
`UNKNOWN` as its own status. Dataset manifests must retain the generator, seed,
split group, complete starting Go state/history, objective, horizon and budgets.
Reference enumeration is intentionally exponential and belongs only on bounded
instances. Larger runs need validated checkpoint/resume before they begin.